# Setup and Connection to a Vector Database

In this notebook we will **set up a vector database**, **open its UI**, **establish a connection** from Python, and **insert some records**.

**Which vector DB?** We use **Qdrant** - an open-source, Rust-based vector database you can run locally (Docker) or in the cloud. It stores vector embeddings and supports fast similarity search.

## Prerequisites

This Docker image starts both Qdrant and JupyterLab together. Inside the notebook, Qdrant is available at `http://localhost:6333`.

### Step 1: Verify Python environment

This Docker image already includes `qdrant-client`. We run a tiny import check so students can see the environment is ready before connecting.

In [ ]:
import qdrant_client
print("qdrant-client imported successfully.")

### Step 2: Import the client

Import `QdrantClient`. We will use this object for all database operations in this demo.

In [ ]:
from qdrant_client import QdrantClient

### Step 3: Connect to Qdrant

Create a client URL from environment variable `QDRANT_URL`. Default is `http://localhost:6333` because Qdrant runs in the same container for this demo.

In [ ]:
import os

qdrant_url = os.getenv("QDRANT_URL", "http://localhost:6333")
client = QdrantClient(url=qdrant_url)
print(f"Client created for: {qdrant_url}")

### Step 4: Verify connection by listing collections

`get_collections()` sends a real request to the server. If Qdrant is up, you should see a response (usually empty initially).

In [ ]:
collections = client.get_collections()
print("Connection established.")
print("Collections:", collections)

### Step 5: Open the Qdrant UI dashboard

Inside a container, `webbrowser.open()` may not open your host browser reliably. So we print a clickable dashboard URL that students can open directly.

In [ ]:
from IPython.display import display, Markdown

dashboard_url = "http://localhost:6333/dashboard"
display(Markdown(f"Open Qdrant Dashboard: [{dashboard_url}]({dashboard_url})"))

### Step 6: Create a collection

A collection defines the vector settings. For this demo, we use small 4-dimensional vectors with cosine distance.

In [ ]:
from qdrant_client.models import Distance, VectorParams

client.recreate_collection(
    collection_name="demo_collection",
    vectors_config=VectorParams(size=4, distance=Distance.COSINE),
)
print("Collection ready: demo_collection")

### Step 7: Insert sample records

Each record (point) has an `id`, a vector, and optional payload metadata. We insert 3 small points for demonstration.

In [ ]:
from qdrant_client.models import PointStruct

client.upsert(
    collection_name="demo_collection",
    points=[
        PointStruct(id=1, vector=[0.1, 0.2, 0.3, 0.4], payload={"topic": "setup"}),
        PointStruct(id=2, vector=[0.2, 0.1, 0.4, 0.3], payload={"topic": "connection"}),
        PointStruct(id=3, vector=[0.9, 0.8, 0.7, 0.6], payload={"topic": "demo"}),
    ],
)
print("Inserted 3 points into demo_collection.")

### Step 8: Verify inserted data

We read back a few points to confirm insertion. Then refresh the dashboard to show the same data in UI.

In [ ]:
result = client.scroll(collection_name="demo_collection", limit=5, with_payload=True)
print(result[0])